In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from akita_model.model import SeqNN

In [3]:
FOLD = 0

# fold 0 - test
# fold 1 - valid
# folds 2-7 - train

In [4]:
TARGET_C = -0.5

# TARGET_C = -10.0

In [5]:
df = pd.read_csv(f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/fold{FOLD}_with_positions_steps.tsv", sep="\t")
# df = pd.read_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_control/fold{FOLD}_with_positions_steps.tsv", sep="\t")

In [6]:
len(df)

46

In [7]:
df

,chrom,fold,PearsonR,centered_start,centered_end,centered_flat_start,centered_flat_end,active_fraction,neutral_fraction,repressive_fraction,ctcf_motif_locs,last_accepted_step
0,chr1,fold0,0.861165,37799936,39110656,192,320,0.391304,0.521739,0.086957,"[(586, 605), (1601, 1620), (6, 25)]",1421
1,chr11,fold0,0.746112,65677312,66988032,195,317,0.333333,0.533333,0.133333,"[(709, 728), (1349, 1368), (1400, 1419), (1487...",1989
2,chr3,fold0,0.670098,38524928,39835648,144,368,0.483871,0.516129,0.000000,"[(218, 237), (1681, 1700)]",1829
3,chr3,fold0,0.672787,53286912,54597632,187,325,0.360000,0.520000,0.120000,"[(950, 969), (1026, 1045), (1424, 1443), (1931...",1999
4,chr3,fold0,0.676442,119885824,121196544,154,358,0.500000,0.500000,0.000000,"[(517, 536), (1148, 1167), (1061, 1080)]",1997
5,chr3,fold0,0.681364,101859328,103170048,202,310,0.371429,0.514286,0.114286,"[(606, 625), (1668, 1687)]",1972
6,chr3,fold0,0.684073,99010560,100321280,163,349,0.476190,0.523810,0.000000,"[(645, 664), (1518, 1537)]",1774
7,chr3,fold0,0.690577,80795648,82106368,159,353,0.473684,0.526316,0.000000,"[(625, 644), (685, 704)]",1874
8,chr3,fold0,0.700263,127893504,129204224,190,322,0.444444,0.555556,0.000000,"[(1275, 1294), (1278, 1297), (1348, 1367)]",611
9,chr3,fold0,0.750709,146692096,148002816,134,378,0.444444,0.518519,0.037037,"[(1225, 1244), (188, 207)]",1999


In [8]:
class OriginalDataset(Dataset):
    def __init__(self, coord_df, init_seq_path):
        self.coords = coord_df
        self.init_seq_path = init_seq_path
    
    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
        X = X.squeeze(0)
        return X

In [9]:
class BoundaryGenerationDataset(Dataset):
    def __init__(self, coord_df, init_seq_path, slice_path, slice=256, cropping=64, bin_size=2048):
        self.coords = coord_df
        self.init_seq_path = init_seq_path
        self.slice_path = slice_path
        self.slice = slice
        self.cropping = cropping
        self.bin_size = bin_size
        
    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]
        
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        X = torch.load(f"{self.init_seq_path}{chrom}_{start}_{end}_X.pt", weights_only=True, map_location=device)
        slice = torch.load(f"{self.slice_path}{chrom}_{start}_{end}_slice.pt", weights_only=True, map_location=device)
        
        edit_start = (self.slice + self.cropping) * self.bin_size
        edit_end = edit_start + self.bin_size
        
        editedX = X.clone()
        editedX[:,:, edit_start:edit_end] = slice
        
        editedX = editedX.squeeze(0)
        
        return editedX

In [10]:
class TriuMatrixDataset(Dataset):
    def __init__(self, coord_df, map_path):
        """
        coord_df: DataFrame with ["chrom", "centered_start", "centered_end"]
        map_path: Directory containing upper-triangle map tensors (e.g. chr1_1000000_1051200_target.pt)
        """
        self.coords = coord_df
        self.map_path = map_path

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]

        file_name = f"{chrom}_{start}_{end}_target.pt"
        file_path = os.path.join(self.map_path, file_name)

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        # Load the flat upper-triangular vector
        triu_tensor = torch.load(file_path, map_location=device)

        # triu_tensor = triu_tensor.squeeze()

        return triu_tensor

In [11]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [12]:
model = SeqNN()
model_path = (
    "/home1/smaruj/pytorch_akita/models/finetuned/mouse/"
    "Hsieh2019_mESC/checkpoints/"
    "Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth"
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [13]:
def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = len(batch_vectors)
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i][0,:]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512]

In [16]:
orig_dataset = OriginalDataset(df, f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X/fold{FOLD}/")
orig_loader = DataLoader(orig_dataset, batch_size=4, shuffle=False)

In [14]:
edited_dataset = BoundaryGenerationDataset(df, 
                                f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X/fold{FOLD}/", 
                                f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/results_repeated/fold{FOLD}/")

edited_loader = DataLoader(edited_dataset, batch_size=4, shuffle=False)

In [17]:
target_dataset = TriuMatrixDataset(df, f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/targets/target_{TARGET_C}/fold{FOLD}/")

target_loader = DataLoader(target_dataset, batch_size=4, shuffle=False)

In [ ]:
# df

In [18]:
slice = 256
cropping = 64
bin_size = 2048

edit_start = (slice + cropping) * bin_size
edit_end = edit_start + bin_size

In [ ]:
edit_start, edit_end

In [19]:
preds_all_orig = []
preds_all_edited = []
targets_all = []
scd_values = []
urq_mean_values = []
og_urq_mean_values = []
target_urq_mean_values = []
edit_counts = []
seq_GC_content = []
slice_GC_content = []
edited_GC_content = []
flat_GC_content = []

# flat_lengths = df["flat_length"].values
counter = 0

model.eval()
with torch.no_grad():
    for orig_batch, edited_batch, target_batch in zip(orig_loader, edited_loader, target_loader):
        orig_batch = orig_batch.to(device)
        edited_batch = edited_batch.to(device)
        target_batch = target_batch.to(device)
        target_batch = target_batch.squeeze(1)
        
        # B, _, L = orig_batch.shape
        # batch_flat_lengths = flat_lengths[counter:counter + B]
        # center = L // 2
        
        # Compute GC content
        gc_content_all = orig_batch[:, 1:3, :].sum(dim=1) / orig_batch.sum(dim=1)  # [B, L]
        gc_content_slice = orig_batch[:, 1:3, edit_start:edit_end].sum(dim=1) / orig_batch[:, :, edit_start:edit_end].sum(dim=1)  # [B, edited_L]
        gc_content_slice_edit = edited_batch[:, 1:3, edit_start:edit_end].sum(dim=1) / edited_batch[:, :, edit_start:edit_end].sum(dim=1)  # [B, edited_L]
        
        # Mean GC content per sequence
        mean_gc_all = gc_content_all.mean(dim=1).cpu().numpy()  # [B]
        mean_gc_slice = gc_content_slice.mean(dim=1).cpu().numpy()  # [B]
        mean_gc_edit = gc_content_slice_edit.mean(dim=1).cpu().numpy()  # [B]
        
        seq_GC_content.extend(mean_gc_all)
        slice_GC_content.extend(mean_gc_slice)
        edited_GC_content.extend(mean_gc_edit)
        
        # # Flat region GC content
        # for i in range(B):
        #     half_len = batch_flat_lengths[i] // 2
        #     start = int(max(center - half_len, 0))
        #     end = int(min(center + half_len, L))
            
        #     orig_slice = orig_batch[i, :, start:end]
        #     orig_gc = orig_slice[1:3].sum() / orig_slice.sum()

        #     flat_GC_content.append(orig_gc.item())
            
        # counter += B
        
        diffs = torch.abs(orig_batch[:, :, edit_start:edit_end] - edited_batch[:, :, edit_start:edit_end])  # shape [B, 4, region_len]
        num_flips = diffs.sum(dim=(1, 2))  # total bit flips per sequence
        num_edits = (num_flips / 2).cpu().numpy()  # divide by 2 to get base edits
        
        edit_counts.extend(num_edits)
        
        preds_orig = model(orig_batch).cpu()
        preds_edited = model(edited_batch).cpu()

        preds_all_orig.extend(preds_orig)
        preds_all_edited.extend(preds_edited)
        targets_all.extend(target_batch)
        
        scd_batch = torch.sqrt(((preds_edited - preds_orig) ** 2).sum(dim=(1, 2)))  # [B]
        scd_values.extend(scd_batch.numpy())
        
        orig_maps = from_upper_triu_batch(preds_orig)
        edited_maps = from_upper_triu_batch(preds_edited)
        
        urq_mean = np.nanmean(edited_maps[:, 0:250, 260:512], axis=(1, 2))
        urq_mean_values.extend(urq_mean)
        
        og_urq_mean = np.nanmean(orig_maps[:, 0:250, 260:512], axis=(1, 2))
        og_urq_mean_values.extend(og_urq_mean)
        
        target_maps = from_upper_triu_batch(target_batch)
        target_urq_mean = np.nanmean(target_maps[:, 0:250, 260:512], axis=(1, 2))
        target_urq_mean_values.extend(target_urq_mean)

/home1/smaruj/miniconda3/envs/pytorch_hic/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv1d(input, weight, bias, self.stride,


In [20]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [21]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [22]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [23]:
pwm_ctcf = np.load("./PWM_with_flanks.npy")

In [24]:
left_flank_slice = pwm_ctcf[:15, :]  
left_flank_transposed = left_flank_slice.T 
left_flank_tensor = torch.tensor(left_flank_transposed, dtype=torch.float32)

In [25]:
right_flank_slice = pwm_ctcf[-15:, :]  
right_flank_transposed = right_flank_slice.T 
right_flank_tensor = torch.tensor(right_flank_transposed, dtype=torch.float32)

In [26]:
motifs_dict = {"CTCF": pwm_CTCF_tensor,
               "left_flank": left_flank_tensor,
               "right_flank": right_flank_tensor}

In [27]:
from memelite import fimo

In [30]:
def rev_comp(seq):
    """Returns the reverse complement of a DNA string."""
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return "".join(complement.get(base, base) for base in reversed(seq))

# Mapping for your OHE
idx_to_base = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}

In [32]:
batch_size = 4

orig_num_CTCFs = []
num_CTCFs = []

orig_CTCFs_coord = []
new_CTCFs_coord = []

avg_orig_fimo_scores = []
avg_new_fimo_scores = []

avg_orig_left_fimo_scores = []
avg_new_left_fimo_scores = []

avg_orig_right_fimo_scores = []
avg_new_right_fimo_scores = []

extra_flank = 60

model.eval()
with torch.no_grad():
    for orig_batch, edited_batch in zip(orig_loader, edited_loader):
        
        orig_batch = orig_batch.to(device)
        edited_batch = edited_batch.to(device)
        
        orig_slice = orig_batch[:, :, edit_start-extra_flank:edit_end+extra_flank]
        edited_slice = edited_batch[:, :, edit_start-extra_flank:edit_end+extra_flank]
        
        # original sequences
        ctcf_orig_hits, left_orig_hits, right_orig_hits = fimo(
            motifs=motifs_dict,
            sequences=orig_slice.cpu(),
            threshold=1e-4,
            reverse_complement=True
        )
        
        # Shift coordinates back to original reference
        ctcf_orig_hits["start"] -= extra_flank
        ctcf_orig_hits["end"]   -= extra_flank
        left_orig_hits["start"] -= extra_flank
        left_orig_hits["end"]   -= extra_flank
        right_orig_hits["start"] -= extra_flank
        right_orig_hits["end"]   -= extra_flank
        
        # edited sequences
        ctcf_edit_hits, left_edit_hits, right_edit_hits = fimo(
            motifs=motifs_dict,
            sequences=edited_slice.cpu(),
            threshold=1e-4,
            reverse_complement=True
        )

        # Shift coordinates back to original reference
        ctcf_edit_hits["start"] -= extra_flank
        ctcf_edit_hits["end"]   -= extra_flank
        left_edit_hits["start"] -= extra_flank
        left_edit_hits["end"]   -= extra_flank
        right_edit_hits["start"] -= extra_flank
        right_edit_hits["end"]   -= extra_flank
        
        for seq_idx in range(batch_size):
            # --- ORIGINAL, CTCF ---
            orig_seq_hits = ctcf_orig_hits[ctcf_orig_hits["sequence_name"] == seq_idx]
            if not orig_seq_hits.empty:
                orig_seq_hits = orig_seq_hits.sort_values(by="start")
                orig_num_CTCFs.append(len(orig_seq_hits))
                orig_ctcf_coords = set(zip(orig_seq_hits["start"], orig_seq_hits["end"], orig_seq_hits["strand"]))
                orig_fimo_score_avg = orig_seq_hits["score"].mean()
                
                print(f"\n--- Batch {seq_idx}: Pre-existing CTCFs ---")
                for _, row in orig_seq_hits.iterrows():
                    # 1. Re-align coordinates to the orig_slice tensor
                    start_idx = int(row['start'] + extra_flank)
                    end_idx = int(row['end'] + extra_flank)
                    
                    # 2. Extract slice from the ORIGINAL tensor
                    hit_ohe = orig_slice[seq_idx, :, start_idx:end_idx]
                    
                    # 3. Convert OHE to string
                    base_indices = hit_ohe.argmax(dim=0).tolist()
                    sequence = "".join([idx_to_base[i] for i in base_indices])
                    
                    # 4. Reverse complement if on the (-) strand for canonical viewing
                    if row['strand'] == '-':
                        sequence = rev_comp(sequence)
                        
                    print(f"Pos: {int(row['start'])} | Strand: {row['strand']} | Seq: {sequence} | Score: {row['score']:.2f}")
                
            else:
                orig_num_CTCFs.append(0)
                orig_ctcf_coords = set()
                orig_fimo_score_avg = 0.0
            orig_CTCFs_coord.append(orig_ctcf_coords)
            avg_orig_fimo_scores.append(orig_fimo_score_avg)
            
            # --- ORIGINAL, LEFT FLANK ---
            orig_left_hits = left_orig_hits[left_orig_hits["sequence_name"] == seq_idx]
            if not orig_left_hits.empty:
                orig_fimo_left_score_avg = orig_left_hits["score"].mean()
            else:
                orig_fimo_left_score_avg = 0.0
            avg_orig_left_fimo_scores.append(orig_fimo_left_score_avg)
                
            # --- ORIGINAL, RIGHT FLANK ---
            orig_right_hits = right_orig_hits[right_orig_hits["sequence_name"] == seq_idx]
            if not orig_right_hits.empty:
                orig_fimo_right_score_avg = orig_right_hits["score"].mean()
            else:
                orig_fimo_right_score_avg = 0.0
            avg_orig_right_fimo_scores.append(orig_fimo_right_score_avg)
            
            # --- EDITED, CTCF ---
            edited_ctcf_seq_hits = ctcf_edit_hits[ctcf_edit_hits["sequence_name"] == seq_idx]
            if not edited_ctcf_seq_hits.empty:
                edited_ctcf_seq_hits = edited_ctcf_seq_hits.sort_values(by="start")
                num_CTCFs.append(len(edited_ctcf_seq_hits))

                # New CTCF sites only
                new_hits = [
                    (start, end, strand, score)
                    for start, end, strand, score in zip(
                        edited_ctcf_seq_hits["start"],
                        edited_ctcf_seq_hits["end"],
                        edited_ctcf_seq_hits["strand"],
                        edited_ctcf_seq_hits["score"]
                    )
                    if (start, end, strand) not in orig_ctcf_coords
                ]
                
                df_new_hits = pd.DataFrame(new_hits, columns=["start", "end", "strand", "score"])
                
                if not df_new_hits.empty:
                    df_new_hits = df_new_hits.sort_values(by="start")
                    new_ctcf_coords = set(zip(df_new_hits["start"], df_new_hits["end"], df_new_hits["strand"]))
                    new_fimo_scores = df_new_hits["score"].mean()
                    
                    # # --- ADDED SECTION: PRINT SEQUENCES ---
                    # for _, row in df_new_hits.iterrows():
                    #     # 1. Pull the slice from the reference (+) strand tensor
                    #     start_idx = int(row['start'] + extra_flank)
                    #     end_idx = int(row['end'] + extra_flank)
                    #     hit_ohe = edited_slice[seq_idx, :, start_idx:end_idx]
                        
                    #     # 2. Convert OHE to string
                    #     base_indices = hit_ohe.argmax(dim=0).tolist()
                    #     sequence = "".join([idx_to_base[i] for i in base_indices])
                        
                    #     # 3. If FIMO found it on the (-) strand, flip it to see the canonical motif
                    #     if row['strand'] == '-':
                    #         sequence = rev_comp(sequence)
                            
                    #     print(f"Batch {seq_idx} | Strand: {row['strand']} | Motif Seq: {sequence}")
                    
                    
                else:
                    new_ctcf_coords = set()
                    new_fimo_scores = 0.0
                new_CTCFs_coord.append(new_ctcf_coords)
                avg_new_fimo_scores.append(new_fimo_scores)
        
            else:
                num_CTCFs.append(0)
                new_CTCFs_coord.append(set())
                avg_new_fimo_scores.append(0.0)   
            
            # --- ORIGINAL, LEFT FLANK ---
            edit_left_hits = left_edit_hits[left_edit_hits["sequence_name"] == seq_idx]
            if not edit_left_hits.empty:
                edit_fimo_left_score_avg = edit_left_hits["score"].mean()
            else:
                edit_fimo_left_score_avg = 0.0
            avg_new_left_fimo_scores.append(edit_fimo_left_score_avg)
                
            # --- ORIGINAL, RIGHT FLANK ---
            edit_right_hits = right_edit_hits[right_edit_hits["sequence_name"] == seq_idx]
            if not edit_right_hits.empty:
                edit_fimo_right_score_avg = edit_right_hits["score"].mean()
            else:
                edit_fimo_right_score_avg = 0.0
            avg_new_right_fimo_scores.append(edit_fimo_right_score_avg)        
                   


--- Batch 0: Pre-existing CTCFs ---
Pos: 6 | Strand: - | Seq: TTGCCGCTAGATGGCAGTG | Score: 18.02
Pos: 586 | Strand: + | Seq: CCGCCAGCAGATGGCGCTA | Score: 21.97
Pos: 1601 | Strand: + | Seq: CGGCCACTAGATGGCGCCG | Score: 22.83

--- Batch 1: Pre-existing CTCFs ---
Pos: 709 | Strand: + | Seq: TATCCAGAAGATGTCGCAA | Score: 13.86
Pos: 1349 | Strand: + | Seq: TCGCCACACGGTGGCGCCA | Score: 15.47
Pos: 1400 | Strand: + | Seq: ATGCCACTAGGTGGCGCCA | Score: 21.30
Pos: 1487 | Strand: + | Seq: CTGCCACCAGGGGGAAGCG | Score: 19.51

--- Batch 2: Pre-existing CTCFs ---
Pos: 218 | Strand: + | Seq: ATGCCACCAGGGGGCGCTC | Score: 21.43
Pos: 1681 | Strand: + | Seq: TCGCCACTAGATGGCGCTG | Score: 22.27

--- Batch 3: Pre-existing CTCFs ---
Pos: 950 | Strand: + | Seq: TGACCACAAGGTGGCAGCA | Score: 21.47
Pos: 1026 | Strand: + | Seq: TGACCACTAGATGGCAGTG | Score: 22.01
Pos: 1424 | Strand: - | Seq: TGGCCAGGAGAGGGCCTTT | Score: 9.79
Pos: 1931 | Strand: - | Seq: GGACCAGCAGGTGGCGCTA | Score: 21.63

--- Batch 0: Pre-existing C

In [ ]:
orig_preds_all = torch.cat(preds_all_orig, dim=0)
edited_preds_all = torch.cat(preds_all_edited, dim=0)
targets_all = torch.cat(targets_all, dim=0)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df["SCD"] = scd_values
df["URQ_result"] = urq_mean_values
df["URQ_target"] = target_urq_mean_values
df["URQ_init"] = og_urq_mean_values
df["num_edits"] = edit_counts
df["GC_seq"] = seq_GC_content
df["GC_slice"] = slice_GC_content
df["GC_slice_edited"] = edited_GC_content
# df["flat_GC_content"] = flat_GC_content

In [ ]:
df["init_CTCFs_num"] = orig_num_CTCFs[:len(df)]
df["CTCFs_num"] = num_CTCFs[:len(df)]
df["avg_orig_fimo_scores"] = avg_orig_fimo_scores[:len(df)]
df["avg_new_fimo_scores"] = avg_new_fimo_scores[:len(df)]

df["orig_CTCFs_coord"] = orig_CTCFs_coord[:len(df)]
df["new_CTCFs_coord"] = new_CTCFs_coord[:len(df)]

df["avg_orig_left_fimo_scores"] = avg_orig_left_fimo_scores[:len(df)]
df["avg_new_left_fimo_scores"] = avg_new_left_fimo_scores[:len(df)]

df["avg_orig_right_fimo_scores"] = avg_orig_right_fimo_scores[:len(df)]
df["avg_new_right_fimo_scores"] = avg_new_right_fimo_scores[:len(df)]

# df["FIMO_sum"] = sum_FIMO[:len(df)]
# df["FIMO_max"] = max_FIMO[:len(df)]
# df["orientation"] = strand_strings[:len(df)]
# df["positions"] = positions[:len(df)]

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df[['URQ_result', 'URQ_target', 'URQ_init']]

In [ ]:
df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/results/fold{FOLD}_with_positions_steps_results.tsv", sep="\t", index=False)
# df.to_csv(f"/scratch1/smaruj/suppressing_CTCFs/results_control/fold{FOLD}_with_positions_steps_results.tsv", sep="\t", index=False)

In [ ]:
batch_size = 4
flank_size = 15
ctcf_ohe_seqs = []  # list to collect OHE motif+flank arrays
ctcf_ohe_scores = [] # FIMO

model.eval()
with torch.no_grad():
    for edited_batch in edited_loader:
        
        edited_batch = edited_batch.to(device)
        edited_slice = edited_batch[:, :, edit_start:edit_end]  # shape: [B, 4, L]
        
        hits = fimo.fimo(
            motifs=motifs_dict,
            sequences=edited_slice,
            threshold=1e-4,
            reverse_complement=True
        )[0]
        
        for seq_idx in range(batch_size):
            seq_hits = hits[hits["sequence_name"] == seq_idx]
            
            if not seq_hits.empty:
                for _, row in seq_hits.iterrows():
                    start = int(row["start"]) - flank_size
                    end = int(row["end"]) + flank_size
                    strand = row["strand"]
                    
                    # Make sure bounds are within slice
                    if start < 0 or end > edited_slice.shape[-1]:
                        continue
                    
                    subseq = edited_slice[seq_idx, :, start:end]  # shape: [4, motif_len+30]
                    
                    if strand == '-':
                        # Reverse sequence
                        subseq = torch.flip(subseq, dims=[-1])  # reverse along the sequence axis

                        # Complement the one-hot rows: A<->T, C<->G
                        complement_map = torch.tensor([3, 2, 1, 0], device=subseq.device)  # old row i becomes new row complement_map[i]
                        subseq = subseq[complement_map, :]
    
                    ctcf_ohe_seqs.append(subseq.cpu().numpy())  # store as numpy array
                    ctcf_ohe_scores.append(row["score"])

In [ ]:
top_indices = np.argsort(ctcf_ohe_scores)[-20:][::-1]  # descending order

In [ ]:
# Extract the top sequences
top_ctcf_ohe = [ctcf_ohe_seqs[i] for i in top_indices]
top_scores = [ctcf_ohe_scores[i] for i in top_indices]

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list

In [ ]:
# Assume top_ctcf_ohe is a list of 20 np.arrays with shape (4, 49)
# Stack them into a single array of shape (20, 4, 49)
seq_array = np.stack(top_ctcf_ohe)  # shape: [20, 4, 49]

# Convert from one-hot to integer-encoded sequences (A=0, C=1, G=2, T=3)
int_encoded = np.argmax(seq_array, axis=1)  # shape: [20, 49]

# Compute pairwise Hamming distances
distance_matrix = squareform(pdist(int_encoded, metric='hamming'))

# Use hierarchical clustering to order sequences
order = leaves_list(linkage(distance_matrix))

# Reorder the sequences
ordered_seqs = int_encoded[order]  # shape: [20, 49]

In [ ]:
# Map base integers to RGB colors
color_map = {
    0: [0.0, 1.0, 0.0],   # A - green
    1: [0.0, 0.0, 1.0],   # C - blue
    2: [1.0, 1.0, 0.0],   # G - yellow
    3: [1.0, 0.0, 0.0],   # T - red
}
rgb_array = np.array([[color_map[base] for base in row] for row in ordered_seqs])  # shape: [20, 49, 3]

In [ ]:
# Plotting
plt.figure(figsize=(12, 6))
plt.imshow(rgb_array, aspect='auto')
plt.title('Ordered CTCF Motifs (by Hamming Distance)')
plt.xlabel('Position')
plt.ylabel('Sequence Index')
plt.xticks(ticks=np.linspace(0, 48, 7), labels=[f'{int(x)}' for x in np.linspace(1, 49, 7)])
plt.yticks(ticks=np.arange(20), labels=[str(i + 1) for i in range(20)])
plt.tight_layout()
plt.show()

### SCD

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["SCD"], bins=30, color='steelblue', edgecolor='black')
plt.xlabel("SCD")
plt.ylabel("Count")
plt.title("Distribution of SCD values")
plt.tight_layout()
plt.show()

In [ ]:
# Make sure scd_values is a list or Series of same length as pivoted
assert len(scd_values) == len(df), "Length mismatch!"

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

state_labels = ["active", "neutral", "repressive"]

for i, label in enumerate(state_labels):
    axes[i].scatter(scd_values, df[f"{label}_fraction"], alpha=0.7)
    axes[i].set_title(f"{label.capitalize()} Fraction vs SCD")
    axes[i].set_xlabel("SCD Value")
    if i == 0:
        axes[i].set_ylabel("Fraction")

plt.tight_layout()
plt.show()

### URQ mean

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(urq_mean_values, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("URQ mean")
plt.ylabel("Count")
plt.title("Distribution of URQ mean values")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(target_urq_mean_values, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Target URQ mean")
plt.ylabel("Count")
plt.title("Distribution of Target URQ mean values")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

In [ ]:
# Compute Pearson R
r_val, p_val = pearsonr(urq_mean_values, target_urq_mean_values)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(urq_mean_values, target_urq_mean_values, alpha=0.6, edgecolors='k')

plt.xlabel("Edited URQ Mean")
plt.ylabel("Target URQ Mean")
plt.title("URQ Mean: Edited vs. Target")

# Annotate with Pearson R
plt.text(0.05, 0.95, f"Pearson R = {r_val:.3f}", transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top')

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Make sure scd_values is a list or Series of same length as pivoted
assert len(urq_mean_values) == len(df), "Length mismatch!"

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

state_labels = ["active", "neutral", "repressive"]

for i, label in enumerate(state_labels):
    axes[i].scatter(urq_mean_values, df[f"{label}_fraction"], alpha=0.7)
    axes[i].set_title(f"{label.capitalize()} Fraction vs URQ mean")
    axes[i].set_xlabel("URQ mean")
    if i == 0:
        axes[i].set_ylabel("Fraction")

plt.tight_layout()
plt.show()

### edit count

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(edit_counts, bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Number of edits / 2048bp-long bin")
plt.ylabel("Count")
plt.title("Distribution of edits number")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df["last_accepted_step_query"], bins=30, color='steelblue', edgecolor='black')
plt.xlabel("Last step with accepted edits")
plt.ylabel("Count")
plt.title("Distribution of last step with accepted edits")
plt.tight_layout()
plt.show()

### GC content

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(edit_counts, seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("number of edits")
plt.ylabel("Seq GC content")
plt.title("number of edits vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["last_accepted_step_query"], seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("last step with accepted edit")
plt.ylabel("Seq GC content")
plt.title("last step with accepted edit vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(urq_mean_values, seq_GC_content, alpha=0.6, edgecolors='k')

plt.xlabel("URQ mean")
plt.ylabel("Seq GC content")
plt.title("URQ mean vs. Seq GC content")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# df.to_csv(f"/scratch1/smaruj/genomic_insertion_loci/fold{FOLD}_{TARGET_C}_all_data.tsv", sep="\t", index=False)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
i = 12

In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(orig_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("before_optimization_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(edited_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("after_optimization_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(edited_preds_all[i,:] - orig_preds_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("difference_map.svg", format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
matrix = from_upper_triu(targets_all[i,:], matrix_len=512, num_diags=2)
plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
# plt.colorbar()
# plt.savefig("target_map.svg", format='svg')
plt.show()

In [ ]:
for i in range(6):
    print(i)
    matrix = from_upper_triu(orig_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(edited_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(edited_preds_all[i,:] - orig_preds_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(22, 26):
    print(i)
    matrix = from_upper_triu(targets_all[i,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.show()

In [ ]:
df